# AlphaEarth Examples — A Tour of Earth's Most Dramatic Changes

This notebook demonstrates **5 interesting examples** of what you can do with AlphaEarth 64-dimensional satellite embeddings:

1. **Deforestation Watch** — Amazon rainforest loss in Pará, Brazil
2. **Glacier Retreat** — Columbia Glacier, Alaska
3. **Spectral Fingerprinting** — What does a city *look like* in 64 dimensions?
4. **Similarity Search** — Find solar farms from a single seed point
5. **Multi-Year Timeline** — Year-by-year change bar chart for any location

Each example is self-contained — feel free to swap in your own coordinates.

**Requirements**: Google Earth Engine account + `pip install earthengine-api geemap numpy matplotlib seaborn`

## Setup

In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Uncomment to authenticate for the first time:
# ee.Authenticate()

ee.Initialize(project='ardent-fusion-421917')

# ── Shared constants ───────────────────────────────────────────────────────────
COLLECTION = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
ALL_YEARS  = list(range(2017, 2025))   # AlphaEarth covers 2017–2024

# False-colour vis params (3 of 64 bands shown as R/G/B)
VIS_RGB = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

# Cosine-dissimilarity vis params (blue = unchanged, red = changed)
VIS_CHANGE = {
    "min": 0, "max": 0.4,
    "palette": ["#2166ac", "#f7f7f7", "#d6604d", "#b2182b"]
}

def get_year_image(year: int) -> ee.Image:
    """Return the annual AlphaEarth mosaic for a given year."""
    return COLLECTION.filter(
        ee.Filter.date(f"{year}-01-01", f"{year+1}-01-01")
    ).mosaic()

def cosine_dissimilarity(img_a: ee.Image, img_b: ee.Image) -> ee.Image:
    """Return pixel-wise cosine dissimilarity (0 = identical, 1 = opposite)."""
    dot  = img_a.multiply(img_b).reduce(ee.Reducer.sum())
    mag_a = img_a.pow(2).reduce(ee.Reducer.sum()).sqrt()
    mag_b = img_b.pow(2).reduce(ee.Reducer.sum()).sqrt()
    similarity = dot.divide(mag_a.multiply(mag_b))
    return ee.Image(1).subtract(similarity).rename('dissimilarity')

print("Setup complete. AlphaEarth collection loaded.")

---
## Example 1 — Deforestation Watch: Amazon Rainforest, Pará, Brazil

The arc of deforestation in the Brazilian Amazon is one of the most well-documented land-use changes on Earth. Here we zoom into Pará state and let the cosine dissimilarity reveal exactly which patches changed between 2017 and 2024.

**What to look for**: Bright red patches = forest cleared in the study period. Straight edges often indicate deliberate clearing; irregular patches may be fire or selective logging.

In [ ]:
# ── Location ───────────────────────────────────────────────────────────────────
LON, LAT, ZOOM = -52.5, -4.8, 11   # Pará, Brazil — active deforestation frontier

img_2017 = get_year_image(2017)
img_2024 = get_year_image(2024)
change   = cosine_dissimilarity(img_2017, img_2024)

m = geemap.Map()
m.set_center(LON, LAT, ZOOM)
m.add_layer(img_2017, VIS_RGB,    '2017 — Amazon (false colour)')
m.add_layer(img_2024, VIS_RGB,    '2024 — Amazon (false colour)')
m.add_layer(change,  VIS_CHANGE, '2017→2024 Cosine Dissimilarity')
m

### Side-by-side split view

In [ ]:
m2 = geemap.Map()
m2.set_center(LON, LAT, ZOOM)
left  = geemap.ee_tile_layer(img_2017, VIS_RGB, '2017')
right = geemap.ee_tile_layer(img_2024, VIS_RGB, '2024')
m2.split_map(left, right)
m2

---
## Example 2 — Glacier Retreat: Columbia Glacier, Alaska

Columbia Glacier has been one of the fastest-retreating glaciers in the world since the 1980s. Over 8 years of AlphaEarth coverage, we can quantify how much the glacier terminus has shifted.

**What to look for**: The exposed rock/sediment (previously under ice) will show strong dissimilarity. Compare with the stable surrounding mountains.

In [ ]:
# ── Location ───────────────────────────────────────────────────────────────────
LON_G, LAT_G, ZOOM_G = -147.05, 61.1, 11   # Columbia Glacier, Prince William Sound, AK

img_2017g = get_year_image(2017)
img_2024g = get_year_image(2024)
change_g  = cosine_dissimilarity(img_2017g, img_2024g)

m_glacier = geemap.Map()
m_glacier.set_center(LON_G, LAT_G, ZOOM_G)
left_g  = geemap.ee_tile_layer(img_2017g, VIS_RGB, '2017 — Columbia Glacier')
right_g = geemap.ee_tile_layer(img_2024g, VIS_RGB, '2024 — Columbia Glacier')
m_glacier.split_map(left_g, right_g)
m_glacier

In [ ]:
m_glacier2 = geemap.Map()
m_glacier2.set_center(LON_G, LAT_G, ZOOM_G)
m_glacier2.add_layer(change_g, VIS_CHANGE, 'Glacier change 2017→2024')
m_glacier2

### Year-by-year mean dissimilarity — how fast is the glacier changing?

In [ ]:
# Build a small bounding box around the glacier terminus
aoi_glacier = ee.Geometry.Rectangle([-147.3, 60.95, -146.8, 61.25])

years   = ALL_YEARS[:-1]   # pairs: (2017,2018), ..., (2023,2024)
dissims = []

for yr in years:
    d = cosine_dissimilarity(get_year_image(yr), get_year_image(yr + 1))
    val = d.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi_glacier,
        scale=30,
        maxPixels=1e8
    ).get('dissimilarity').getInfo()
    dissims.append(val)
    print(f"  {yr}→{yr+1}: {val:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([f"{y}→{y+1}" for y in years], dissims, color='steelblue', edgecolor='white')
ax.set_title('Columbia Glacier — Annual Mean Cosine Dissimilarity', fontsize=13)
ax.set_xlabel('Year pair')
ax.set_ylabel('Mean dissimilarity (0 = no change)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## Example 3 — Spectral Fingerprinting: What Does a City Look Like in 64 Dimensions?

AlphaEarth encodes *all* land-surface information into 64 numbers. By sampling a handful of well-known locations and plotting their embedding vectors, we can answer: **do similar land-cover types cluster together in embedding space?**

We sample five sites:
- Dense urban core (Manhattan)
- Desert (Sahara, Libya)
- Tropical forest (Amazon)
- Active farmland (Iowa cornbelt)
- Open ocean (mid-Atlantic)

Then we plot a heatmap of all 64 embedding dimensions for each site.

In [ ]:
SITES = {
    'Dense urban\n(Manhattan)':   (-73.98, 40.75),
    'Desert\n(Sahara)':           (13.50, 25.00),
    'Tropical forest\n(Amazon)':  (-60.00, -3.50),
    'Farmland\n(Iowa)':           (-93.50, 42.00),
    'Open ocean\n(mid-Atlantic)': (-35.00, 30.00),
}

img_2024_fp = get_year_image(2024)
band_names  = [f'A{i:02d}' for i in range(64)]

vectors = {}
for name, (lon, lat) in SITES.items():
    pt = ee.Geometry.Point([lon, lat])
    raw = img_2024_fp.sample(region=pt, scale=100, numPixels=1).first().toDictionary()
    info = raw.getInfo()
    vec  = np.array([info.get(b, 0.0) for b in band_names])
    vectors[name] = vec
    print(f"Sampled {name.replace(chr(10), ' ')}: norm={np.linalg.norm(vec):.3f}")

# Build matrix: sites × bands
matrix = np.vstack(list(vectors.values()))  # (5, 64)

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    matrix,
    ax=ax,
    xticklabels=[f'A{i:02d}' for i in range(0, 64, 4)] + [''],
    yticklabels=list(vectors.keys()),
    cmap='RdBu_r',
    center=0,
    linewidths=0.3,
    cbar_kws={'label': 'Embedding value'}
)
ax.set_title('AlphaEarth 64-band Spectral Fingerprints — Five Land Cover Types (2024)', fontsize=13)
ax.set_xlabel('Embedding dimension')
plt.xticks(ticks=range(0, 64, 4), labels=[f'A{i:02d}' for i in range(0, 64, 4)], rotation=45)
plt.tight_layout()
plt.show()

### Pairwise cosine similarity between sites

In [ ]:
labels = list(vectors.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

for i, (name_i, vi) in enumerate(vectors.items()):
    for j, (name_j, vj) in enumerate(vectors.items()):
        cos = np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj) + 1e-12)
        sim_matrix[i, j] = cos

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    sim_matrix,
    ax=ax,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt='.3f',
    cmap='YlOrRd',
    vmin=0.5, vmax=1.0
)
ax.set_title('Pairwise Cosine Similarity Between Land Cover Types', fontsize=12)
plt.tight_layout()
plt.show()

---
## Example 4 — Similarity Search: Find Solar Farms from a Single Seed Point

One of the most powerful uses of embeddings is **similarity search**: given the embedding vector of one known object, find all other pixels in a region with a similar vector.

Here we seed from a **large solar farm in the Mojave Desert** (Ivanpah Solar Electric Generating System, California) and search the surrounding 150 km for pixels with similar embeddings. Solar farms have a highly distinctive multi-spectral signature that AlphaEarth captures well.

> Try changing the `SEED_LON`/`SEED_LAT` to any land-cover feature you want to find!

In [ ]:
# ── Seed: Ivanpah Solar Farm, Mojave Desert ────────────────────────────────────
SEED_LON, SEED_LAT = -115.476, 35.559
SEARCH_RADIUS_DEG  = 1.3          # ~145 km search box half-width
SIMILARITY_THRESHOLD = 0.92       # pixels above this are 'similar'

img_search = get_year_image(2024)

# Sample the seed point embedding
seed_point = ee.Geometry.Point([SEED_LON, SEED_LAT])
seed_vals  = img_search.sample(region=seed_point, scale=30, numPixels=1).first()
seed_image = ee.Image.constant(
    [seed_vals.get(f'A{i:02d}') for i in range(64)]
).rename([f'A{i:02d}' for i in range(64)])

# Compute cosine similarity everywhere in the search box
search_box = ee.Geometry.Rectangle([
    SEED_LON - SEARCH_RADIUS_DEG, SEED_LAT - SEARCH_RADIUS_DEG,
    SEED_LON + SEARCH_RADIUS_DEG, SEED_LAT + SEARCH_RADIUS_DEG,
])

dot_s    = img_search.multiply(seed_image).reduce(ee.Reducer.sum())
mag_img  = img_search.pow(2).reduce(ee.Reducer.sum()).sqrt()
mag_seed = seed_image.pow(2).reduce(ee.Reducer.sum()).sqrt()
sim_map  = dot_s.divide(mag_img.multiply(mag_seed)).rename('similarity')

# Threshold to highlight matches
matches = sim_map.updateMask(sim_map.gt(SIMILARITY_THRESHOLD))

m_solar = geemap.Map()
m_solar.set_center(SEED_LON, SEED_LAT, 9)
m_solar.add_layer(img_search.clip(search_box), VIS_RGB, 'AlphaEarth 2024 (false colour)')
m_solar.add_layer(
    matches.clip(search_box),
    {"min": SIMILARITY_THRESHOLD, "max": 1.0,
     "palette": ["#ffffb2", "#fd8d3c", "#bd0026"]},
    f'Similar to Ivanpah (>{SIMILARITY_THRESHOLD:.0%})'
)

# Mark seed point
m_solar.add_layer(
    ee.Image().paint(seed_point.buffer(2000), 1).selfMask(),
    {'palette': ['#00ff00']}, 'Seed point'
)

m_solar

---
## Example 5 — Multi-Year Timeline: Year-by-Year Change Bar Chart

Instead of just comparing start-to-end, we can plot **annual change rates** to see *when* change happened. This is useful for detecting sudden events (fires, floods, construction bursts) vs. gradual processes (urban creep, vegetation die-off).

We compare three very different sites:
- **Camp Fire scar** — Paradise, CA (catastrophic wildfire Nov 2018)
- **Las Vegas suburban sprawl** — fast urban growth
- **Greenland ice sheet edge** — slow, steady cryosphere change

In [ ]:
SITES_TIMELINE = {
    'Camp Fire scar\n(Paradise, CA)':    ee.Geometry.Rectangle([-121.7, 39.7, -121.4, 39.9]),
    'Las Vegas sprawl\n(Henderson, NV)': ee.Geometry.Rectangle([-115.2, 35.8, -114.8, 36.2]),
    'Greenland ice edge\n(SW coast)':    ee.Geometry.Rectangle([-51.5, 67.0, -50.5, 67.8]),
}

year_pairs = [(y, y+1) for y in ALL_YEARS[:-1]]  # 7 pairs

results = {name: [] for name in SITES_TIMELINE}

for name, aoi in SITES_TIMELINE.items():
    print(f"\nComputing {name.replace(chr(10), ' ')}...")
    for yr_a, yr_b in year_pairs:
        d = cosine_dissimilarity(get_year_image(yr_a), get_year_image(yr_b))
        val = d.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi,
            scale=30,
            maxPixels=1e8
        ).get('dissimilarity').getInfo()
        results[name].append(val if val is not None else 0.0)
        print(f"  {yr_a}→{yr_b}: {val:.4f}")

In [ ]:
x_labels = [f"{a}→{b}" for a, b in year_pairs]
colors   = ['#e74c3c', '#2ecc71', '#3498db']

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

for ax, (name, vals), color in zip(axes, results.items(), colors):
    bars = ax.bar(x_labels, vals, color=color, alpha=0.85, edgecolor='white', linewidth=0.8)
    ax.set_title(name.replace('\n', ' — '), fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean dissimilarity')
    ax.set_ylim(0, max(vals) * 1.35)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # Annotate the peak year
    peak_idx = int(np.argmax(vals))
    ax.annotate(
        f'Peak: {x_labels[peak_idx]}\n({vals[peak_idx]:.3f})',
        xy=(peak_idx, vals[peak_idx]),
        xytext=(peak_idx + 0.4, vals[peak_idx] * 1.08),
        fontsize=9,
        arrowprops=dict(arrowstyle='->', color='black'),
    )

axes[-1].set_xlabel('Year pair')
axes[-1].tick_params(axis='x', rotation=30)

fig.suptitle('Annual AlphaEarth Dissimilarity — Three Contrasting Sites', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../images/annual_dissimilarity_three_sites.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to images/annual_dissimilarity_three_sites.png")

### Interpreting the results

| Site | Expected peak | Why |
|---|---|---|
| Camp Fire scar | **2018→2019** | Wildfire burned ~240 km² in November 2018; satellite sees scorched landscape in 2019 annual composite |
| Las Vegas sprawl | Relatively steady | Urban growth is continuous, not episodic — no single dramatic year |
| Greenland ice edge | Variable | Melt season intensity varies year-to-year; anomalous melt years (e.g. 2019) may stand out |

> Note: Year labels show the *first* year of each pair. If GEE returns `None` for a cell it means the region lacked cloud-free imagery — treat it as missing data.

---
## Bonus — Band Importance: Which of the 64 Dimensions Changes Most?

Not all 64 embedding bands contribute equally to change signals. Let's compute the **per-band standard deviation of change** (2017 vs 2024) over a region of interest and see which dimensions are most informative.

In [ ]:
# Use the Amazon deforestation region from Example 1
AOI_BONUS = ee.Geometry.Rectangle([-53.0, -5.2, -52.0, -4.4])

img_a = get_year_image(2017)
img_b = get_year_image(2024)

# Per-band absolute difference
diff = img_b.subtract(img_a).abs()

# Mean per band over AOI
stats = diff.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=AOI_BONUS,
    scale=30,
    maxPixels=1e8
).getInfo()

band_names_64 = [f'A{i:02d}' for i in range(64)]
importance    = np.array([stats.get(b, 0.0) for b in band_names_64])

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(range(64), importance, color=plt.cm.plasma(importance / importance.max()))
ax.set_xticks(range(0, 64, 4))
ax.set_xticklabels([f'A{i:02d}' for i in range(0, 64, 4)], rotation=45)
ax.set_xlabel('Embedding dimension')
ax.set_ylabel('Mean |2024 − 2017|')
ax.set_title('Per-band Change Magnitude — Amazon Deforestation Region (2017→2024)', fontsize=13)

# Highlight top 5
top5 = np.argsort(importance)[-5:]
for idx in top5:
    ax.bar(idx, importance[idx], color='red', alpha=0.9)
    ax.text(idx, importance[idx] + 0.001, f'A{idx:02d}', ha='center', va='bottom', fontsize=7, color='red')

plt.tight_layout()
plt.show()
print(f"Top 5 most-changed dimensions: {[f'A{i:02d}' for i in sorted(top5)]}")

---
## Summary

| Example | Technique | Key insight |
|---|---|---|
| 1 — Amazon deforestation | Cosine dissimilarity map | Change pixel outlines cleared forest patches precisely |
| 2 — Columbia Glacier | Annual dissimilarity bar chart | Retreat rate varies year-to-year; highest-change years track warm summers |
| 3 — Spectral fingerprinting | 64-band heatmap + similarity matrix | Urban, desert, ocean, and forest have highly distinctive signatures |
| 4 — Solar farm search | Per-pixel cosine similarity to seed | One seed point finds candidate sites across 150 km |
| 5 — Multi-site timeline | Year-by-year bar charts | Camp Fire shows sharp 2018→2019 spike; sprawl shows gradual rise |
| Bonus — Band importance | Per-band mean absolute difference | Only a subset of dimensions carries most of the change signal |

### Next steps
- Apply a **Gaussian spatial smoothing** kernel (see `AlphaEarth_Story.ipynb`) before thresholding to reduce salt-and-pepper noise
- Combine with **LandTrendr** NDVI outputs to validate detections (see `AlphaEarth_LandTrendr_ChangeComparison.ipynb`)
- Export change masks to Google Drive for GIS analysis